# Mutability and Hashability of Custom Class Objects

**Pillars in play:** E ● A ● I ○ P ○
**Companions:** State and immutability ● · Contracts and LSP ● · Coupling and cohesion ○

**Why this matters.** Every `set`, every `dict` key, every cache key (`lru_cache`, Redis keys you build from objects), and every "is this the same order?" check relies on two dunder methods, `__eq__` and `__hash__`, and on whether the object can change. Interviewers ask *"Why does defining `__eq__` make a class unhashable?"* because the answer shows whether you understand the **hash contract**, **identity vs equality**, and **value objects vs entities**.

How the topic touches each pillar:

- **Encapsulation**: immutability is the strongest form of encapsulation. If nothing can change the state, no one can break an invariant.
- **Abstraction**: a value object exposes *what it is* (amount + currency), not storage details.
- **Inheritance**: subclasses can silently lose `__hash__`, which is an LSP trap.
- **Polymorphism**: `Hashable` is a protocol. Any object honouring it works as a dict key, whatever its class.

> **How to use this notebook.** Cells marked 🔮 are *predict first*: write your prediction **and the reason** before running. Outputs are intentionally not saved.

### Contents
1. Mental model: two independent axes
2. The default state of a fresh class
3. Instance state is mutable by default (the machinery)
4. Defining `__eq__` sets `__hash__` to `None`
5. Restoring hashability: value objects vs entities (and the broken way)
6. Inheritance rules for `__eq__` / `__hash__`
7. Making classes immutable: the toolkit, from hand-written to `dataclass(frozen=True)`
8. The dataclass hash matrix
9. Shallow vs deep immutability
10. Gotcha gallery
11. Testing the contract (assert helper + Hypothesis)
12. Framework link: Django, Pydantic, SQLAlchemy
13. Engineering guidelines (decision table)
14. Tasks (warm-up, core, stretch)
15. Interview Q&A
16. Pillar Check and cheat sheet

In [ ]:
import sys
from contextlib import contextmanager

print(sys.version)


@contextmanager
def expect(*exceptions: type[BaseException]):
    # Tiny helper: run a block that is EXPECTED to raise, print the error, keep the notebook going.
    try:
        yield
    except exceptions as exc:
        print(f"💥 {type(exc).__name__}: {exc}")
    else:
        print("✅ no exception")

## 1. Mental model: two independent axes

**Mutability** asks whether the object's observable state can change after creation.
**Hashability** asks whether the object has a `__hash__` that is consistent with its `__eq__` and stable for as long as the object sits in a hash-based container.

They are *related but not the same*. The rule that ties them together:

> **The hash contract**
> 1. `a == b` ⟹ `hash(a) == hash(b)` (the reverse is NOT required: collisions are allowed).
> 2. `hash(x)` must not change while `x` is inside a `set` or used as a `dict` key.
>
> Corollary: **hash only what `__eq__` compares, and only what cannot change.**

Why rule 2 matters (how a set/dict lookup works):

```
x in s
  │
  ├─ 1. h = hash(x)            → pick a bucket from h
  ├─ 2. for each entry e in that bucket's probe chain:
  │        if e is x            → found  (identity shortcut!)
  │        if e.hash == h and e == x → found
  └─ 3. otherwise              → not found
```

The bucket is chosen **when the item is inserted**. If the hash later changes, lookups go to the wrong bucket, and the object is "lost" in the container.

| | hashable | unhashable |
|---|---|---|
| **immutable** | `int`, `str`, `tuple` of hashables, `frozenset`, frozen dataclass | `tuple` containing a `list` (immutable shell, mutable content) |
| **mutable** | **plain custom class (default!)**, entity with stable id | `list`, `dict`, `set`, class with `__eq__` only |

The bold cell is the surprising one, and it's where custom classes start.

🔮 **Predict:** which of these are hashable? Why is `(1, [2])` special?

In [ ]:
from collections.abc import Hashable

samples = [42, "tenant-1", (1, 2), (1, [2]), [1, 2], frozenset({1}), {1}, None, object()]
for s in samples:
    try:
        hash(s)
        ok = "hashable"
    except TypeError as exc:
        ok = f"unhashable ({exc})"
    print(f"{s!r:<22} isinstance Hashable={isinstance(s, Hashable)!s:<5} → {ok}")

Note `(1, [2])`: `isinstance(..., Hashable)` says **True** (the `tuple` *class* defines `__hash__`), yet `hash()` fails because the tuple hashes its items. `Hashable` checks only that the method exists, not that it works. The same thing happens with frozen dataclasses holding lists (section 9).

## 2. The default state of a fresh class

A class with no dunder methods inherits `__eq__` and `__hash__` from `object`:

- `object.__eq__` → returns `True` if `self is other`, otherwise `NotImplemented`.
- `==` then tries the reflected `other.__eq__(self)`; if that is also `NotImplemented`, Python falls back to **identity** (`a is b`).
- `object.__hash__` → derived from `id(self)`.

Equality is identity, and the hash is identity-based, so the contract holds trivially. Identity never changes during an object's lifetime, so **mutating state cannot break the hash**.

🔮 **Predict:** `a == b`? `hash(a) == hash(b)`? How many items in `{a, b}`?

In [ ]:
class Order:
    def __init__(self, order_id: int, status: str = "new") -> None:
        self.order_id = order_id
        self.status = status


a = Order(1)
b = Order(1)  # same data, different object

print("a == b       :", a == b)
print("a is b       :", a is b)
print("hash equal   :", hash(a) == hash(b))
print("len({a, b})  :", len({a, b}))

**Internals peek:** where do these methods come from?

In [ ]:
print("__eq__ defined on Order? ", "__eq__" in vars(Order))
print("__hash__ defined on Order?", "__hash__" in vars(Order))
print("Order.__eq__ is object.__eq__    :", Order.__eq__ is object.__eq__)
print("Order.__hash__ is object.__hash__:", Order.__hash__ is object.__hash__)

print("object.__eq__(a, b) →", object.__eq__(a, b))  # NotImplemented, not False!
print("object.__eq__(a, a) →", object.__eq__(a, a))

# CPython implementation detail (not a language guarantee): the hash is the address rotated by 4 bits
print("hash(a) == id(a) >> 4:", hash(a) == id(a) >> 4)

🔮 **Predict:** we put `a` in a set and a dict, then mutate it. Is it still found?

In [ ]:
active_orders = {a}
by_order = {a: "warehouse-7"}

a.status = "paid"      # mutate state
a.order_id = 999       # even the "id"!

print(a in active_orders)
print(by_order[a])

Yes, it's still found. Default objects are **mutable and hashable, and that's safe**, because neither `==` nor `hash` reads the state. This is the classic **entity** semantics: *"it's the same object, whatever it currently contains"*.

The cost: two `Order(1)` objects loaded from the DB in two places are **not equal**. Whether that's correct depends on the domain (see section 5).

## 3. Instance state is mutable by default: the machinery

A regular instance keeps its state in a per-instance `dict` (`obj.__dict__`). Assignment, addition and deletion are just dict operations, and **there is no schema**.

🔮 **Predict:** does `id(o)` change? What does `vars(o)` contain at the end?

In [ ]:
o = Order(7)
before = id(o)

o.status = "shipped"   # mutate existing attribute
o.note = "fragile"     # add a brand-new attribute: allowed!
o.stauts = "lost"      # typo → silently creates ANOTHER attribute
del o.note             # delete

print(vars(o))
print("same object:", id(o) == before)

Two lessons:

1. **Mutation keeps identity.** The object is the same, only its `__dict__` changed. Rebinding (`o = Order(8)`) is different: it points the *name* at another object.
2. **The typo bug.** `o.stauts = ...` raises no error. `__slots__` (section 7) removes `__dict__` and turns this into an `AttributeError`.

**Aliasing**: mutation is visible through *every* reference. This is why shared mutable objects are risky in caches, default arguments and across threads.

In [ ]:
o1 = Order(10)
o2 = o1            # NOT a copy, a second name for the same object
o2.status = "cancelled"
print(o1.status, o1 is o2)

**Encapsulation leak**: returning internal mutable state lets callers bypass your invariants.

In [ ]:
class Cart:
    MAX_ITEMS = 3

    def __init__(self) -> None:
        self.items: list[str] = []          # public & mutable

    def add(self, sku: str) -> None:
        if len(self.items) >= self.MAX_ITEMS:  # invariant guarded HERE…
            raise ValueError("cart is full")
        self.items.append(sku)


cart = Cart()
cart.items.extend(["a", "b", "c", "d", "e"])  # …and bypassed HERE
print(len(cart.items), "items, invariant broken")


class SafeCart:
    MAX_ITEMS = 3

    def __init__(self) -> None:
        self._items: list[str] = []

    @property
    def items(self) -> tuple[str, ...]:      # expose an immutable VIEW
        return tuple(self._items)

    def add(self, sku: str) -> None:
        if len(self._items) >= self.MAX_ITEMS:
            raise ValueError("cart is full")
        self._items.append(sku)


safe = SafeCart()
safe.add("a")
with expect(AttributeError):
    safe.items.append("b")  # tuple has no append

## 4. Defining `__eq__` sets `__hash__` to `None`

When a class body defines `__eq__` but **not** `__hash__`, `type.__new__` inserts `__hash__ = None` into the class namespace.

Why: once you switch equality to *values*, the inherited identity hash would break rule 1 (two equal objects with different hashes). Python won't guess a correct hash for you, so it disables hashing and makes you decide.

🔮 **Predict:** what is `Money.__hash__`? What happens with `{Money(...)}`? And with `Money(...) in [Money(...)]`?

In [ ]:
from decimal import Decimal


class Money:
    def __init__(self, amount: Decimal, currency: str) -> None:
        self.amount = amount
        self.currency = currency

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Money):
            return NotImplemented          # let the other operand try, don't return False
        return (self.amount, self.currency) == (other.amount, other.currency)

    def __repr__(self) -> str:
        return f"Money({self.amount!r}, {self.currency!r})"


m1, m2 = Money(Decimal("10"), "EUR"), Money(Decimal("10"), "EUR")

print("Money.__hash__            :", Money.__hash__)
print("'__hash__' in vars(Money) :", "__hash__" in vars(Money), "→", vars(Money)["__hash__"])
print("isinstance(m1, Hashable)  :", isinstance(m1, Hashable))
print("m1 == m2, m1 != m2        :", m1 == m2, m1 != m2)   # __ne__ is derived from __eq__ automatically
print("m1 in [m2]                :", m1 in [m2])           # lists use __eq__ only, no hash needed

with expect(TypeError):
    {m1}
with expect(TypeError):
    {m1: "price"}

`list` membership uses only `__eq__` (a linear scan), so it still works. `set` and `dict` need `__hash__` first. This is also why an "unhashable" error only shows up *at the place where you hash*, not where you define the class.

## 5. Restoring hashability: value objects vs entities

There are three legitimate options and two broken ones:

| Strategy | `__eq__` compares | `__hash__` uses | State must be… | Domain meaning |
|---|---|---|---|---|
| **Value object** | all defining fields | same fields | **immutable** | "equal if same value" (`Money`, `Email`, `DateRange`) |
| **Entity** | identity field (`id`) | `id` only | `id` immutable; rest may change | "same thing over time" (`Tenant`, `User`, `Order`) |
| **Opt out** | values | `__hash__ = None` | mutable | value-comparable but not a key (`Cart`) |
| ❌ hash mutable fields | values | mutable fields | mutable | **objects get lost in sets/dicts** |
| ❌ `__hash__ = object.__hash__` with value `__eq__` | values | identity | – | **contract broken**: equal objects in two buckets |

### 5a. ❌ The broken way first: hashing mutable state

🔮 **Predict all five lines.** The last two are the interesting ones.

In [ ]:
class MutableMoney:
    def __init__(self, amount: int, currency: str) -> None:
        self.amount = amount
        self.currency = currency

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, MutableMoney):
            return NotImplemented
        return (self.amount, self.currency) == (other.amount, other.currency)

    def __hash__(self) -> int:
        return hash((self.amount, self.currency))   # hashes MUTABLE fields


m = MutableMoney(10, "EUR")
wallet = {m}
prices = {m: "coffee"}

m.amount = 99   # hash(m) changes, but m stays in the OLD bucket

print("1. m in wallet                 :", m in wallet)
print("2. m physically inside wallet  :", any(x is m for x in wallet))
with expect(KeyError):
    print(prices[m])
print("4. MutableMoney(10,'EUR') in w :", MutableMoney(10, "EUR") in wallet)
wallet.add(m)
print("5. len(wallet) after re-adding :", len(wallet), "→ the SAME object twice:", [x is m for x in wallet])

Line 4 is subtle: the old bucket matches by hash, but `==` now compares against `99`, so it fails too. The object can't be found **by any key**. Line 5 breaks the core invariant of a set (no duplicates). In production this looks like "cache never hits", "dedup doesn't dedup" or "KeyError for a key that is clearly there".

### 5b. ❌ Identity hash + value equality

In [ ]:
class Sku:
    def __init__(self, code: str) -> None:
        self.code = code

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Sku) and self.code == other.code

    __hash__ = object.__hash__   # "just make it hashable again"


x, y = Sku("A-1"), Sku("A-1")
print(x == y, hash(x) == hash(y), len({x, y}))   # equal, yet 2 elements in a set

### 5c. ✅ Value object: immutable + hash the same fields as `__eq__`

Hand-written first, to see every moving part (section 7 shows the one-liner).

In [ ]:
class Money:
    __slots__ = ("_amount", "_currency")          # no __dict__: no new attributes, less memory

    def __init__(self, amount: Decimal, currency: str) -> None:
        # our __setattr__ blocks assignment, so bypass it ONCE, here, on purpose
        object.__setattr__(self, "_amount", amount)
        object.__setattr__(self, "_currency", currency.upper())

    @property
    def amount(self) -> Decimal:
        return self._amount

    @property
    def currency(self) -> str:
        return self._currency

    def __setattr__(self, name: str, value: object) -> None:
        raise AttributeError(f"{type(self).__name__} is immutable")

    def __delattr__(self, name: str) -> None:
        raise AttributeError(f"{type(self).__name__} is immutable")

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Money):
            return NotImplemented
        return (self._amount, self._currency) == (other._amount, other._currency)

    def __hash__(self) -> int:
        return hash((self._amount, self._currency))  # SAME fields as __eq__

    def __repr__(self) -> str:
        return f"Money({self._amount!r}, {self._currency!r})"


price = Money(Decimal("9.90"), "eur")
print({price, Money(Decimal("9.90"), "EUR")})          # deduplicated
print({price: "coffee"}[Money(Decimal("9.90"), "EUR")])  # found by an equal key

with expect(AttributeError):
    price.amount = Decimal("0")   # property without setter
with expect(AttributeError):
    price._amount = Decimal("0")  # our __setattr__
with expect(AttributeError):
    price.discount = 5            # __setattr__ (and no __dict__ anyway)

> **Python immutability is a contract, not a vault.** `object.__setattr__(price, "_amount", ...)` still works. We protect against *accidents*, not against someone who is deliberately bypassing us ("consenting adults"). That's enough for correctness; security belongs elsewhere.

### 5d. ✅ Entity: equality and hash by an immutable identity

The rest of the state stays mutable. This is how a multi-tenant system treats a `Tenant`.

In [ ]:
from uuid import UUID, uuid4


class Tenant:
    __slots__ = ("_id", "name", "plan")

    def __init__(self, tenant_id: UUID, name: str, plan: str = "free") -> None:
        self._id = tenant_id
        self.name = name        # mutable on purpose
        self.plan = plan        # mutable on purpose

    @property
    def id(self) -> UUID:       # read-only: identity must never change
        return self._id

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Tenant):
            return NotImplemented
        return self._id == other._id

    def __hash__(self) -> int:
        return hash(self._id)

    def __repr__(self) -> str:
        return f"Tenant({self._id!s:.8}…, {self.name!r}, {self.plan!r})"


tid = uuid4()
t = Tenant(tid, "Acme")
seen = {t}

t.plan = "pro"                     # mutate non-identity state
t.name = "Acme Corp"
print(t in seen)                   # still found: hash didn't depend on name/plan
print(Tenant(tid, "stale copy from cache") == t)  # same identity → equal
with expect(AttributeError):
    t.id = uuid4()

### 5e. ✅ Explicit opt-out

If a class is value-comparable but mutable, say so explicitly. It documents intent and also blocks a hash inherited from a parent.

In [ ]:
class Basket:
    def __init__(self, *skus: str) -> None:
        self.skus = list(skus)

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Basket) and sorted(self.skus) == sorted(other.skus)

    __hash__ = None   # explicit: "comparable by value, NOT a dict key" (also what Python would do implicitly)


print(Basket("a", "b") == Basket("b", "a"))
with expect(TypeError):
    {Basket("a")}

## 6. Inheritance rules for `__eq__` / `__hash__`

The "`__eq__` without `__hash__` → `__hash__ = None`" rule is applied **per class body**, including subclasses.

| Subclass body defines | Result in subclass |
|---|---|
| nothing | inherits parent's `__eq__` and `__hash__` |
| only `__hash__` | new hash, parent's `__eq__` |
| only `__eq__` | **`__hash__ = None`** even if parent was hashable |
| both | both new |

🔮 **Predict** each line before running.

In [ ]:
class Base:
    def __init__(self, key: str) -> None:
        self.key = key

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Base) and self.key == other.key

    def __hash__(self) -> int:
        return hash(self.key)


class Inherits(Base):
    pass


class OverridesEq(Base):
    def __eq__(self, other: object) -> bool:      # e.g. "just adding logging"
        return super().__eq__(other)


class OverridesEqKeepsHash(Base):
    def __eq__(self, other: object) -> bool:
        return super().__eq__(other)

    __hash__ = Base.__hash__                      # explicitly re-attach


for cls in (Base, Inherits, OverridesEq, OverridesEqKeepsHash):
    print(f"{cls.__name__:<22} __hash__ = {cls.__hash__}")

**LSP angle.** Code that stores `Base` objects in a `set` breaks the moment someone passes an `OverridesEq` object. The subclass is **not substitutable**: a "harmless" `__eq__` override silently changed the contract. Name it in a review: *"this override violates LSP because the base contract includes hashability."*

**Symmetry and subclasses.** With `isinstance` checks, `Base("x") == Child("x")` can be `True` even when `Child` adds fields, which gives asymmetric or non-transitive equality. Dataclasses avoid this by comparing only when `other.__class__ is self.__class__` (otherwise they return `NotImplemented`):

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Price:
    amount: int


@dataclass(frozen=True)
class DiscountedPrice(Price):
    discount: int = 0


print(Price(10) == DiscountedPrice(10))   # different classes → not equal
print(Price(10) == Price(10))

## 7. Making classes immutable: the toolkit

| Technique | Blocks assignment | Blocks new attrs | Hash for free | Validation hook | Cost / gotcha |
|---|---|---|---|---|---|
| read-only `@property` over `_x` | public names only | no | no | `__init__` | boilerplate, `_x` still writable |
| `__slots__` alone | **no** | yes | no | `__init__` | not immutability (see gotcha G9) |
| `__slots__` + `__setattr__`/`__delattr__` | yes | yes | write it | `__init__` | most boilerplate (5c) |
| `typing.NamedTuple` | yes | yes | yes | ✗ (needs `__new__`) | **is a tuple**: `== (1, 2)`, iterable, indexable |
| `@dataclass(frozen=True, slots=True)` | yes | yes | yes | `__post_init__` | **modern default for domain value objects** |
| Pydantic `ConfigDict(frozen=True)` | yes | yes | yes | validators | for boundaries (API/settings); heavier |
| `attrs.frozen` | yes | yes | yes | validators | third-party; slots by default |

### 7a. `NamedTuple`: immutable, but it leaks its tuple-ness

🔮 **Predict:** `GeoPoint(50.08, 14.43) == (50.08, 14.43)`?

In [ ]:
from typing import NamedTuple


class GeoPoint(NamedTuple):
    lat: float
    lon: float


class ScreenSize(NamedTuple):
    width: float
    height: float


prague = GeoPoint(50.08, 14.43)
print(hash(prague) == hash((50.08, 14.43)))
print(prague == (50.08, 14.43))          # equal to ANY tuple with same items…
print(GeoPoint(1, 2) == ScreenSize(1, 2))  # …even an unrelated NamedTuple
lat, lon = prague                         # unpackable, indexable, len()…: leaky abstraction
with expect(AttributeError):
    prague.lat = 0
print(prague._replace(lat=0.0))

Use `NamedTuple` for lightweight records where tuple behaviour is a *feature* (DB rows, `csv` lines, multiple return values). For domain values, the leaked tuple API weakens **abstraction**.

### 7b. `@dataclass(frozen=True, slots=True)`: the modern default

In [ ]:
import copy
import dataclasses
from dataclasses import FrozenInstanceError, dataclass, field
from typing import Self


@dataclass(frozen=True, slots=True, kw_only=True)
class Money:
    amount: Decimal
    currency: str

    def __post_init__(self) -> None:
        # validation: the invariant is checked once, at the only entry point
        if self.amount < 0:
            raise ValueError("amount must be non-negative")
        # normalisation needs the escape hatch: frozen __setattr__ raises
        object.__setattr__(self, "currency", self.currency.upper())

    def __add__(self, other: object) -> Self:
        if not isinstance(other, Money):
            return NotImplemented
        if other.currency != self.currency:
            raise ValueError("currency mismatch")
        return dataclasses.replace(self, amount=self.amount + other.amount)  # new object, never mutate


fee = Money(amount=Decimal("2.50"), currency="eur")
total = fee + Money(amount=Decimal("7.50"), currency="EUR")
print(fee, total, sep="\n")
print({fee, Money(amount=Decimal("2.50"), currency="EUR")})   # hashable + deduped

with expect(FrozenInstanceError):
    fee.amount = Decimal("0")
with expect(FrozenInstanceError, AttributeError, TypeError):
    fee.extra = 1
with expect(ValueError):
    Money(amount=Decimal("-1"), currency="EUR")

# "Modification" = make a new object
print(dataclasses.replace(fee, amount=Decimal("3")))
print(copy.replace(fee, amount=Decimal("4")))   # Python 3.13+: generic copy.replace()
print("FrozenInstanceError is an AttributeError:", issubclass(FrozenInstanceError, AttributeError))

**Internals peek:** what did the decorator generate?

In [ ]:
print("params     :", Money.__dataclass_params__)
print("__slots__  :", Money.__slots__)
print("has __dict__ on instance:", hasattr(fee, "__dict__"))
print("__hash__   :", Money.__hash__)   # generated: hashes the tuple of compared fields
print("__setattr__:", Money.__setattr__)
print("fields     :", [(f.name, f.compare, f.hash) for f in dataclasses.fields(Money)])

**Excluding fields from equality and hash.** Metadata like `loaded_at` shouldn't affect identity. Use `field(compare=False)`, which also removes it from the hash. `hash=None` (the default) means "follow `compare`".

In [ ]:
from datetime import datetime, UTC


@dataclass(frozen=True, slots=True)
class ExchangeRate:
    base: str
    quote: str
    rate: Decimal
    fetched_at: datetime = field(default_factory=lambda: datetime.now(UTC), compare=False)


r1 = ExchangeRate("EUR", "CZK", Decimal("25.1"))
r2 = ExchangeRate("EUR", "CZK", Decimal("25.1"))
print(r1.fetched_at != r2.fetched_at or "same µs", "| equal:", r1 == r2, "| same hash:", hash(r1) == hash(r2))

### 7c. Pydantic v2 at the boundary

In [ ]:
from pydantic import BaseModel, ConfigDict, ValidationError


class TokenClaims(BaseModel):
    model_config = ConfigDict(frozen=True)
    sub: str
    scopes: tuple[str, ...]          # tuple, not list: keeps the hash working


class MutableDTO(BaseModel):         # default Pydantic model
    sub: str


claims = TokenClaims(sub="user-1", scopes=["orders:read"])   # list coerced to tuple by validation
print(claims, "| hash ok:", isinstance(hash(claims), int))
print("default BaseModel __hash__:", MutableDTO.__hash__)
with expect(ValidationError):
    claims.sub = "admin"
print(claims.model_copy(update={"sub": "user-2"}))  # careful: model_copy(update=) does NOT re-validate

### 7d. `attrs` (awareness)

In [ ]:
import attrs


@attrs.frozen   # = define(frozen=True): slots=True by default, eq + hash generated
class Email:
    value: str = attrs.field(converter=str.lower)


print(Email("Denys@Example.COM"), Email("a@b.c") == Email("A@B.C"), len({Email("x@y.z"), Email("X@Y.Z")}))

## 8. The dataclass hash matrix

`@dataclass` decides `__hash__` from `eq`, `frozen` and `unsafe_hash` (when the class body doesn't define `__hash__` itself). Let's generate the matrix instead of memorising it:

In [ ]:
from itertools import product


def describe_hash(cls: type) -> str:
    h = cls.__dict__.get("__hash__", "<not set>")
    if h is None:
        return "None → unhashable"
    if h == "<not set>":
        return "inherited object.__hash__ (identity)"
    return "generated from fields"


print(f"{'eq':<6}{'frozen':<8}{'unsafe_hash':<13}→ __hash__")
for eq, frozen, unsafe in product((True, False), repeat=3):
    if frozen and not eq:
        pass  # allowed, just unusual
    ns = {"__annotations__": {"x": int}}
    cls = dataclass(eq=eq, frozen=frozen, unsafe_hash=unsafe)(type("D", (), ns))
    print(f"{eq!s:<6}{frozen!s:<8}{unsafe!s:<13}→ {describe_hash(cls)}")

Read it as:

- `@dataclass` (eq=True, not frozen) → **unhashable**: same rule as a hand-written `__eq__`.
- `frozen=True` → field-based hash. **This is the one you want for values.**
- `eq=False` → identity semantics, like a plain class (even with `frozen=True`).
- `unsafe_hash=True` on a mutable class → field hash on mutable fields, i.e. **the broken pattern from 5a**. The name is a warning label. A legitimate use is rare (e.g. you *guarantee* by convention that the object is never mutated after creation, but can't freeze it because a library needs to set attributes).

## 9. Shallow vs deep immutability

`frozen=True` freezes **the attribute bindings**, not the objects they point to.

🔮 **Predict:** does construction fail? Does `append` work? Does `hash()` work?

In [ ]:
@dataclass(frozen=True)
class OrderSnapshot:
    order_id: int
    lines: list[str]


snap = OrderSnapshot(1, ["sku-1"])          # construction: fine
snap.lines.append("sku-2")                  # mutating the list: fine (!)
print(snap)
with expect(TypeError):
    hash(snap)                              # fails only NOW, maybe far away in a cache

The error surfaces late, at the place that hashes, possibly in a library or under load. Fail early instead: use immutable field types, and **copy the caller's input** so the caller can't keep a handle on your internals.

In [ ]:
from collections.abc import Iterable


@dataclass(frozen=True, slots=True)
class OrderSnapshot:
    order_id: int
    lines: tuple[str, ...]
    tags: frozenset[str] = frozenset()

    def __post_init__(self) -> None:
        # defensive conversion: accept any iterable, store an immutable copy
        object.__setattr__(self, "lines", tuple(self.lines))
        object.__setattr__(self, "tags", frozenset(self.tags))


caller_lines = ["sku-1"]
snap = OrderSnapshot(1, caller_lines, {"vip"})
caller_lines.append("sku-HACK")             # caller mutates their own list later
print(snap.lines, snap.tags, hash(snap) is not None)

Immutable replacements for containers:

| Mutable | Immutable + hashable | Note |
|---|---|---|
| `list` | `tuple` | |
| `set` | `frozenset` | |
| `dict` | `tuple(sorted(d.items()))` or `frozenset(d.items())` | `types.MappingProxyType` is a *read-only view*, **not** hashable and not a copy. A built-in `frozendict` has been proposed (PEP 814); check the docs for your Python version. |
| custom mutable object | its frozen counterpart | the freeze is only as deep as the weakest field |

## 10. Gotcha gallery

Predict each one first, then run it. Pillar tags are in brackets.

### G1. Augmented assignment on a frozen field: error *and* mutation [E]

🔮 **Predict:** after the `FrozenInstanceError`, what is `snap.lines`?

In [ ]:
@dataclass(frozen=True)
class Draft:
    lines: list[str]


d = Draft(["a"])
with expect(FrozenInstanceError):
    d.lines += ["b"]     # 1) list.__iadd__ mutates in place  2) then setattr → raises
print(d.lines)

`x += y` is `x = x.__iadd__(y)`. The in-place mutation succeeds *before* the assignment is rejected. Same with `t = ([],); t[0] += [1]`.

### G2. Class-level mutable attribute is shared state [E]

In [ ]:
class Notifier:
    sent: list[str] = []          # ONE list on the class, shared by all instances

    def send(self, msg: str) -> None:
        self.sent.append(msg)     # mutates the class attribute (no assignment → no shadowing)


n1, n2 = Notifier(), Notifier()
n1.send("hi")
print(n2.sent, "| shared:", n1.sent is n2.sent is Notifier.sent)

### G3. Mutable default in a dataclass [E]

🔮 **Predict:** which of the three definitions raise, and when, at class creation or at instantiation?

In [ ]:
with expect(ValueError):
    @dataclass
    class A:
        tags: list[str] = []                 # rejected at class creation

with expect(ValueError):
    @dataclass
    class B:
        meta: Basket = Basket()              # rejected too: the check is "is the default unhashable?" (3.11+)


@dataclass
class C:
    tags: list[str] = field(default_factory=list)   # ✅ new list per instance


print(C().tags is not C().tags)

Note the logic: dataclasses use *unhashable* as a proxy for *mutable*. So a mutable-but-hashable default (a plain object instance) is **not** caught, and it would be shared between instances.

### G4. `cached_property` vs frozen / slots [E, version-sensitive]

In [ ]:
from functools import cached_property


@dataclass(frozen=True)
class Report:
    rows: tuple[int, ...]

    @cached_property
    def total(self) -> int:     # writes into instance __dict__ directly, bypassing __setattr__
        return sum(self.rows)


@dataclass(frozen=True, slots=True)
class SlottedReport:
    rows: tuple[int, ...]

    @cached_property
    def total(self) -> int:
        return sum(self.rows)


print("frozen          :", Report((1, 2, 3)).total)
with expect(TypeError, AttributeError):
    print("frozen + slots :", SlottedReport((1, 2, 3)).total)   # no __dict__ to cache into

`cached_property` works with `frozen=True` (it writes to `__dict__` directly) but not with `slots=True`. If you need both, compute the value in `__post_init__` into a `field(init=False, compare=False)`, or skip caching.

### G5. Zero-arg `super()` inside `slots=True` dataclass [I, version-sensitive]

In [ ]:
@dataclass
class Event:
    name: str

    def describe(self) -> str:
        return f"event {self.name}"


@dataclass(slots=True)
class PaymentEvent(Event):
    amount: int = 0

    def describe(self) -> str:
        return super().describe() + f" ({self.amount})"


print(sys.version_info[:2])
with expect(TypeError):
    print(PaymentEvent("paid", 10).describe())

`slots=True` must **create a new class**, because slots can't be added to an existing class. For a long time methods kept their `__class__` cell pointing at the *old* class, so zero-arg `super()` raised `TypeError` (CPython issue gh-90562). **Verified on this kernel (3.13.14): it works.** On older interpreters it fails, so if your code must support them, use `super(PaymentEvent, self)` or declare `__slots__` by hand. The lesson for interviews: *`slots=True` returns a different class object than the one you wrote*, and anything that captured the original class (closures, registries filled in `__init_subclass__`) can still see the old one.

### G6. Identity shortcut in containers [P]

🔮 **Predict:** `nan == nan`? `nan in [nan]`? `nan in {nan}`?

In [ ]:
nan = float("nan")
print(nan == nan, nan in [nan], nan in {nan})


class AlwaysUnequal:
    def __eq__(self, other: object) -> bool:
        return False

    __hash__ = object.__hash__


u = AlwaysUnequal()
print(u == u, u in [u], u in {u})     # containers check `is` BEFORE `==`

Containers assume reflexivity (`x == x`) and check identity first. An `__eq__` that isn't reflexive gives inconsistent results between `==` and `in`.

### G7. Cross-type equality collapses dict keys [A, P]

In [ ]:
d = {1: "int", True: "bool", 1.0: "float", Decimal(1): "decimal"}
print(d)          # 1 == True == 1.0 == Decimal(1), equal hashes → ONE key

This is why a value object's `__eq__` should usually check the type (`isinstance` or same-class) and return `NotImplemented` for foreign types, instead of comparing loosely.

### G8. `hash()` is not stable across processes [backend!]

In [ ]:
import hashlib
import os
import subprocess

snippet = "print(hash('tenant-42:/orders'))"
for seed in ("1", "2"):
    out = subprocess.run([sys.executable, "-c", snippet], capture_output=True, text=True,
                         env={**os.environ, "PYTHONHASHSEED": seed})
    print(f"PYTHONHASHSEED={seed}: {out.stdout.strip()}")

# stable alternative for Redis keys, ETags, idempotency keys:
print(hashlib.sha256(b"tenant-42:/orders").hexdigest()[:16])

`str`/`bytes` hashes are randomized per process (hash-flooding protection). **Never** persist `hash()` or use it for Redis keys shared between workers. Use `hashlib` on a canonical representation.

### G9. `__slots__` ≠ immutable [E]

In [ ]:
class Point:
    __slots__ = ("x", "y")

    def __init__(self, x: int, y: int) -> None:
        self.x, self.y = x, y


p = Point(1, 2)
p.x = 100                         # still mutable
with expect(AttributeError):
    p.z = 3                       # only NEW attributes are blocked
print(p.x, "| hashable (identity):", Point.__hash__ is object.__hash__)

### G10. `lru_cache` on a method: hash requirement + memory leak [E]

In [ ]:
from functools import lru_cache


@dataclass                           # eq=True → unhashable
class PricingService:
    vat: Decimal

    @lru_cache(maxsize=128)          # cache key includes `self`
    def gross(self, net: Decimal) -> Decimal:
        return net * (1 + self.vat)


with expect(TypeError):
    PricingService(Decimal("0.21")).gross(Decimal("100"))

Even when `self` *is* hashable, the cache holds strong references to every `self`, so instances never get freed. Prefer a module-level cached function with hashable arguments, or `cached_property` for per-instance values.

## 11. Testing the contract

Treat `__eq__`/`__hash__` as a **contract with tests**, like any other public API. A reusable helper:

In [ ]:
def assert_value_object(a: object, equal_to_a: object, different: object) -> None:
    assert a is not equal_to_a, "pass two distinct but equal objects"
    # equality: reflexive, symmetric, distinguishes
    assert a == a and a == equal_to_a and equal_to_a == a
    assert a != different
    # hash contract rule 1
    assert hash(a) == hash(equal_to_a)
    # works as key / set member, found via an EQUAL (not identical) object
    assert {a: "v"}[equal_to_a] == "v"
    assert len({a, equal_to_a, different}) == 2
    # immutability: no public field can be reassigned
    for name in [n for n in dir(a) if not n.startswith("_")]:
        if callable(getattr(a, name)):
            continue
        try:
            setattr(a, name, None)
        except (AttributeError, TypeError):
            pass
        else:
            raise AssertionError(f"field {name!r} is assignable")


assert_value_object(
    Money(amount=Decimal("1"), currency="EUR"),
    Money(amount=Decimal("1.00"), currency="eur"),
    Money(amount=Decimal("2"), currency="EUR"),
)
print("✅ Money satisfies the value-object contract")

try:
    assert_value_object(MutableMoney(1, "EUR"), MutableMoney(1, "EUR"), MutableMoney(2, "EUR"))
except AssertionError as exc:
    print("❌ MutableMoney:", exc)

**Property-based** (Hypothesis): generate many inputs and check the invariant holds for all of them.

In [ ]:
from hypothesis import given, strategies as st

amounts = st.decimals(min_value=0, max_value=10**9, allow_nan=False, allow_infinity=False, places=2)
currencies = st.sampled_from(["EUR", "eur", "CZK", "czk", "USD"])


@given(amounts, currencies, amounts, currencies)
def test_eq_implies_equal_hash(a1, c1, a2, c2):
    m1, m2 = Money(amount=a1, currency=c1), Money(amount=a2, currency=c2)
    if m1 == m2:
        assert hash(m1) == hash(m2)


@given(amounts, currencies)
def test_add_never_mutates_operands(a, c):
    m = Money(amount=a, currency=c)
    before = (m.amount, m.currency)
    _ = m + m
    assert (m.amount, m.currency) == before


test_eq_implies_equal_hash()
test_add_never_mutates_operands()
print("✅ hypothesis: properties hold")

## 12. Framework link

**Django models are entities with a pk-based hash.** From `django/db/models/base.py` (abridged; read it in your installed Django with `inspect.getsource(Model.__hash__)`):

```python
def __eq__(self, other):
    if not isinstance(other, Model):
        return NotImplemented
    if self._meta.concrete_model != other._meta.concrete_model:
        return False
    my_pk = self.pk
    if my_pk is None:
        return self is other          # unsaved: fall back to identity
    return my_pk == other.pk

def __hash__(self):
    if self.pk is None:
        raise TypeError("Model instances without primary key value are unhashable")
    return hash(self.pk)
```

This is section 5d exactly: identity = pk, other fields mutable. An unsaved instance refuses to be hashed, because its hash *would change* on `save()` (rule 2). Practical consequence: `{Order()}` before saving → `TypeError`; two querysets loading the same row give `==` objects.

**Pydantic v2:** `BaseModel` is unhashable by default (`__hash__` is `None`, as shown in 7c). `frozen=True` generates a field-based hash. Use frozen models for settings and token claims; keep request/response DTOs mutable unless you need them as keys.

**SQLAlchemy 2.0:** the Session's **identity map** guarantees one Python object per row per session, so identity-based `==`/hash (the default) is already "by primary key" within a session. Overriding `__eq__`/`__hash__` on mapped classes with mutable columns can break collections and relationship bookkeeping. Leave the defaults unless you have a clear reason.

**Pillar question:** which pillar does Django's `Model.__hash__` demonstrate? (Encapsulation of the identity invariant + a contract every subclass inherits: LSP.)

## 13. Engineering guidelines (2026 defaults)

| You are modelling… | Use | Eq / hash |
|---|---|---|
| Domain **value** (`Money`, `Email`, `DateRange`, `CacheKey`) | `@dataclass(frozen=True, slots=True)` (+ `kw_only=True` for 3+ fields) | by all fields (generated) |
| Domain **entity** (`Tenant`, `Order`, `User`) | plain class or `@dataclass(eq=False)`; immutable `id` via read-only property | by `id`, or keep identity default |
| API boundary / settings | Pydantic v2 `BaseModel` (`frozen=True` for settings/claims) | per config |
| Lightweight record / multiple returns | `NamedTuple` | tuple semantics (accept the leak) |
| Mutable aggregate (`Cart`, `UnitOfWork`) | plain class, encapsulated state | identity, or `__hash__ = None` if value-compared |

Rules of thumb (**convention** unless marked):

1. **Freeze by default; mutate with a reason.** Immutable objects are easier to reason about, safe to share between threads and async tasks without locks (more relevant with free-threaded builds), and safe as cache keys.
2. **"Change" = new object.** `dataclasses.replace` / `copy.replace` (3.13+) / `model_copy`. Return `Self` from "with_*" methods.
3. **Hash ⊆ eq fields, and only immutable ones** (*fact*: the contract).
4. **Deep-freeze fields**: `tuple`/`frozenset`, defensive copies in `__post_init__`.
5. **`__eq__` returns `NotImplemented`** for foreign types, never `False` (*fact*: enables reflected comparison).
6. **Never use `unsafe_hash=True` or `__hash__ = object.__hash__`** to "fix" a TypeError. Find out whether the class is a value or an entity.
7. **Never persist `hash()`.** Use `hashlib` for anything that leaves the process.
8. **Test the contract** (section 11) for every value object.
9. Typing: annotate keys as `collections.abc.Hashable` when you need "any hashable"; mypy flags assignment to frozen dataclass fields and `Final` attributes at check time.

## 14. Tasks

Solve these yourself. The test cell below each stub must print ✅. Hint ladder: ask for **H1 → H4** only after a real attempt.

### Warm-up (★, ~5 min): `Notification`

`Notification` already compares by `(recipient, channel)`. Make it a proper **hand-written** value object (no dataclass):
1. Hashable, consistent with `__eq__`.
2. After `__init__`, no attribute can be assigned, added or deleted (including `_recipient`).
3. No instance `__dict__`.

In [ ]:
class Notification:
    def __init__(self, recipient: str, channel: str) -> None:
        self._recipient = recipient
        self._channel = channel

    @property
    def recipient(self) -> str:
        return self._recipient

    @property
    def channel(self) -> str:
        return self._channel

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Notification):
            return NotImplemented
        return (self._recipient, self._channel) == (other._recipient, other._channel)

    # TODO: make it hashable and immutable (you may change __init__)

In [ ]:
# ── tests: do not edit ──
def _raises(exc, fn):
    try:
        fn()
    except exc:
        return True
    return False


n = Notification("denys@example.com", "email")
assert len({n, Notification("denys@example.com", "email"), Notification("denys@example.com", "sms")}) == 2
assert {n: 1}[Notification("denys@example.com", "email")] == 1
assert _raises(AttributeError, lambda: setattr(n, "recipient", "x"))
assert _raises(AttributeError, lambda: setattr(n, "_recipient", "x")), "private field must be protected too"
assert _raises(AttributeError, lambda: setattr(n, "extra", 1))
assert _raises(AttributeError, lambda: delattr(n, "_channel"))
assert not hasattr(n, "__dict__"), "use __slots__"
assert n.recipient == "denys@example.com"
print("✅ warm-up passed")

### Core 1 (★★, ~15 min): `Money` for the online-cinema billing

Write `Money` as a **frozen, slotted dataclass**.

- Fields: `amount: Decimal`, `currency: str`.
- `amount` must be a `Decimal` (reject `float` with `TypeError`, since floats are imprecise for money) and `>= 0` (`ValueError`).
- `currency` is normalised to upper case and must be 3 letters (`ValueError`).
- `a + b` → new `Money`; currency mismatch → `ValueError`; non-`Money` operand → `TypeError` (hint: what should `__add__` *return*?).
- `with_amount(amount) -> Self` returns a new object; the original is unchanged.
- Must pass `assert_value_object` from section 11.

In [ ]:
@dataclass  # TODO: options
class Money:
    amount: Decimal
    currency: str

    def __post_init__(self) -> None:
        raise NotImplementedError

    def __add__(self, other: object) -> Self:
        raise NotImplementedError

    def with_amount(self, amount: Decimal) -> Self:
        raise NotImplementedError

In [ ]:
# ── tests: do not edit ──
assert dataclasses.is_dataclass(Money)
assert Money.__dataclass_params__.frozen, "must be frozen"
assert "__slots__" in vars(Money), "must use slots"

m = Money(Decimal("10.00"), "eur")
assert m.currency == "EUR"
assert _raises(TypeError, lambda: Money(10.0, "EUR"))
assert _raises(ValueError, lambda: Money(Decimal("-1"), "EUR"))
assert _raises(ValueError, lambda: Money(Decimal("1"), "EURO"))
assert _raises(ValueError, lambda: Money(Decimal("1"), "E1R"))

assert m + Money(Decimal("5"), "EUR") == Money(Decimal("15.00"), "EUR")
assert _raises(ValueError, lambda: m + Money(Decimal("5"), "CZK"))
assert _raises(TypeError, lambda: m + 5)

m2 = m.with_amount(Decimal("1"))
assert m2 == Money(Decimal("1"), "EUR") and m.amount == Decimal("10.00") and m2 is not m

assert_value_object(m, Money(Decimal("10"), "EUR"), Money(Decimal("11"), "EUR"))
print("✅ core 1 passed")

### Core 2 (★★, ~15 min): `CacheKey` for a rate-limited public endpoint

Responses are cached per `(tenant_id, path, query params)`. Query params arrive as a `dict`.

- Frozen dataclass with fields `tenant_id: int`, `path: str`, `params: tuple[tuple[str, str], ...]`.
- Alternate constructor `CacheKey.from_request(tenant_id, path, params: Mapping[str, str]) -> Self`.
- Param **order must not matter**: `{"a": "1", "b": "2"}` and `{"b": "2", "a": "1"}` give equal keys with equal hashes.
- Mutating the original dict afterwards must not affect the key.
- `fingerprint() -> str`: a 64-char hex string, **stable across processes** (it will become a Redis key).

In [ ]:
from collections.abc import Mapping


@dataclass  # TODO
class CacheKey:
    tenant_id: int
    path: str
    params: tuple[tuple[str, str], ...]

    @classmethod
    def from_request(cls, tenant_id: int, path: str, params: Mapping[str, str]) -> Self:
        raise NotImplementedError

    def fingerprint(self) -> str:
        raise NotImplementedError

In [ ]:
# ── tests: do not edit ──
q = {"page": "2", "sort": "-created"}
k1 = CacheKey.from_request(42, "/orders", q)
k2 = CacheKey.from_request(42, "/orders", {"sort": "-created", "page": "2"})
k3 = CacheKey.from_request(43, "/orders", q)

assert isinstance(k1, Hashable) and isinstance(hash(k1), int)
assert k1 == k2 and hash(k1) == hash(k2), "param order must not matter"
assert k1 != k3
assert {k1: "cached"}[k2] == "cached"

q["page"] = "999"
assert k1 == k2, "mutating the source dict must not change the key"

fp = k1.fingerprint()
assert len(fp) == 64 and int(fp, 16) >= 0
assert fp == k2.fingerprint() and fp != k3.fingerprint()
assert _raises(AttributeError, lambda: setattr(k1, "path", "/admin"))
print("✅ core 2 passed")

### Stretch (★★★): `Subscription` entity, Django-style

- Mutable `plan: str` and `status: str`; `id: UUID | None` starts as `None` (not yet persisted).
- `assign_id(uuid)` sets it **once**; a second call raises `RuntimeError`. No other way to change `id` (read-only property).
- `__eq__`: same class and both ids set and equal; if either id is `None`, fall back to identity.
- `__hash__`: `TypeError` if `id is None`; else hash of `id`.
- Mutating `plan`/`status` must not affect set membership.

**Written part** (answer in a markdown cell, aloud in English, ≤ 90 s): *Why not `@dataclass(frozen=True)` here? Why must `__hash__` refuse while `id` is `None` instead of falling back to `id(self)`?*

In [ ]:
class Subscription:
    def __init__(self, plan: str, status: str = "active") -> None:
        raise NotImplementedError

In [ ]:
# ── tests: do not edit ──
s = Subscription("basic")
assert _raises(TypeError, lambda: hash(s)), "unsaved → unhashable"
assert s == s and s != Subscription("basic"), "unsaved → identity equality"

uid = uuid4()
s.assign_id(uid)
assert _raises(RuntimeError, lambda: s.assign_id(uuid4()))
assert _raises(AttributeError, lambda: setattr(s, "id", uuid4()))

bucket = {s}
s.plan, s.status = "premium", "past_due"
assert s in bucket, "mutating non-identity state must not lose the entity"

twin = Subscription("stale")
twin.assign_id(uid)
assert twin == s and hash(twin) == hash(s)
print("✅ stretch passed")

*Your written answer here:*

## 15. Interview Q&A

Answer **out loud first** (60–90 s), then open the model answer.

**Q1. Why does defining `__eq__` make a class unhashable, and how do you fix it?**

<details><summary>Model answer</summary>

By default, objects compare and hash by identity, so the hash contract (equal objects must have equal hashes) holds trivially. When I define `__eq__`, I change what "equal" means, and the inherited identity hash would break the contract: two equal objects would land in different buckets. Python can't guess the right hash, so at class creation it sets `__hash__` to `None`. The fix depends on the domain. For a **value object**, I make it immutable, usually `@dataclass(frozen=True, slots=True)`, and hash the same fields that `__eq__` compares. For an **entity**, I compare and hash by an immutable identity such as the primary key, and the rest can stay mutable; that's exactly what Django's `Model.__hash__` does. What I don't do is restore `object.__hash__` or use `unsafe_hash`, because that only hides the contract violation.
</details>

**Q2. Can a mutable object be hashable? When is that OK?**

<details><summary>Model answer</summary>

Yes. Every plain class instance is mutable and hashable by default. It's safe because the hash and equality depend only on identity, which never changes. The rule is that the hash must not change while the object sits in a set or dict, so a mutable object is fine as a key as long as its hash depends only on things that can't change: its identity, or an immutable id field. It becomes a bug when the hash is computed from mutable fields. After a mutation the object is in the wrong bucket, lookups fail, and you can even get the same object twice in a set.
</details>

**Q3. Frozen dataclass vs NamedTuple vs frozen Pydantic model for a domain value object?**

<details><summary>Model answer</summary>

For a domain value object, my default is a frozen, slotted dataclass. It's immutable, generates `__eq__` and `__hash__`, is light, and `__post_init__` gives me one place for validation. `NamedTuple` is also immutable and hashable, but it *is* a tuple: it equals any tuple with the same items and is iterable and indexable, so it leaks its representation. That's fine for records, not for domain types. A frozen Pydantic model is the right tool at boundaries such as API payloads, settings or token claims, where I need parsing and serialization, but it's heavier and couples the domain to a library. In all three cases immutability is shallow, so I use tuples or frozensets for collection fields.
</details>

## 16. Pillar Check and cheat sheet

Answer before asking for feedback (one or two sentences each, naming the principle):

- **E:** In your `Money`, what state is protected, and which invariant guards it? Where is the *single* place the invariant is checked?
- **A:** What is the public surface of `CacheKey`, and what complexity (canonicalisation, hashing) is hidden behind it?
- **I / composition:** In section 6, why is overriding only `__eq__` in a subclass an LSP violation?
- **P:** Your `Money`, `CacheKey` and Django models all work as dict keys. What is the shared contract that makes this polymorphism?

### Cheat sheet

```text
fresh class         eq = identity, hash = identity      → mutable + hashable, SAFE
define __eq__ only  → __hash__ = None                   → unhashable (per class body, incl. subclasses)
value object        frozen + hash(same fields as eq)    → @dataclass(frozen=True, slots=True)
entity              eq/hash by immutable id             → rest mutable; Django Model pattern
opt out             __hash__ = None                     → explicit "not a key"
NEVER               hash mutable fields / unsafe_hash / __hash__ = object.__hash__ with value eq
contract            a == b ⇒ hash(a) == hash(b); hash stable while in a container
frozen is SHALLOW   use tuple / frozenset, defensive copy in __post_init__
"change" a value    dataclasses.replace / copy.replace (3.13+) / model_copy
across processes    hashlib, never hash()
```